# Modelo de **Hodgkin & Huxley**

Referencias:

1. https://en.wikipedia.org/wiki/Hodgkin%E2%80%93Huxley_model
2. Eugene M. Izhikevich, Dynamical Systems in Neuroscience: The Geometry of Excitability and Bursting, Computational Neuroscience, The MIT Press (2006)
3. https://colab.research.google.com/github/kmkarakaya/ML_tutorials/blob/master/How_to_upload_an_image_to_Colab_.ipynb#scrollTo=KKTpNAGCWLeB

## **Ejercicio 1)** Importando librerÃ­as

**a)** Importe las librerÃ­as `numpy` para operar con arrays, `scipy` para utilizar rutinas de algebra lineal y `matplotlib.pyplot` para graficar.

**b)** Suba el archivo "libreria" `ode.py` que fabricÃ³ en el prÃ¡ctico 4), e impÃ³rtelo.

In [ ]:
# 1.a)
import numpy as np
import scipy as sp
import scipy.linalg as linalg
import matplotlib.pyplot as plt

In [ ]:
# 1.b)
# https://stackoverflow.com/questions/48905127/importing-py-files-in-google-colab
#import ode

def euler(f,x,t,h,p):
    """
    Calcula un paso de integraciÃ³n del mÃ©todo de Euler.

    Argumentos de entrada:

        f : R^n,R -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numÃ©rica de

        x(t+h) : R^n

    segÃºn el mÃ©todo de Euler.

    # Ejemplos:
    """
    return x+h*f(x,t,p)

def rk4(f,x,t,h,p):
    """
    Calcula un paso de integraciÃ³n del mÃ©todo de Runge Kutta orden 4.

    Argumentos de entrada:

        f : R^n -> R^n
        x = x(t) : R^n
        t = tiempo : R
        h = paso de tiempo : R
        p = parametros : R^q

    Retorna aproximacion numÃ©rica de

        x(t+h) : R^n

    segÃºn el mÃ©todo de Runge-Kutta de orden 4.

    # Ejemplos:
    """
    k1 = f(x,t,p)
    k2 = f(x+0.5*h*k1,t+0.5*h,p)
    k3 = f(x+0.5*h*k2,t+0.5*h,p)
    k4 = f(x+h*k3,t+h,p)
    return x+h*(k1+2.0*k2+2.0*k3+k4)/6.0

def integrador_ode(m,f,xa,a,b,k,p,c=lambda x,t,p:x):
    """
    Integra numÃ©ricamente la ODE

        dx/dt = f(x,t)

    sobre el intervalo t:[a,b] usando k pasos de integraciÃ³n y el mÃ©todo m, bajo condicion inicial x(a)=x0.
    No es necesario que a<b.

    Argumentos de entrada:

        m = metodo de integracion (ej. euler, rk2, etc.)
        f : R^n -> R^n
        xa = condicion inicial : R
        a = tiempo inicial : R
        b = tiempo final : R
        k = num. pasos de integracion : N
        p = parametros : R^q
        c = funciÃ³n condicionante : R^n,R,p -> R^n

    Retorna:

        t : R^{k+1} , t_j = a+j*h para j=0,1,...,k
        w : R^{n,k+1} , w_ij = x_i(t_j) para i=0,1,...,n-1 y j=0,1,...,k

    donde a+k*dt = b.
    """
    assert k>0
    n = len(xa)
    h = (b-a)/k
    w = np.zeros((n,k+1)) # Produce un array con forma y tipo especificada con los parametros,
                          # lleno de ceros. la forma puede ser espcificada con un entero o tupla (n,k+1)
    t = np.zeros(k+1)
    w[:,0] = xa           # actualiza la posicion inicial (columna de indice 0) de las variables con los valores
                          # de las condiciones iniciales
    t[0] = a              # actualiza la posicion cero con el valor del tiempo inicial

    for j in range(k):    #Aca se produce la iteraciÃ³n en j

        t[j+1] = t[j] + h                # iteracion tiempo
        w[:,j+1] = m(f,w[:,j],t[j],h,p)  # iteracion de w
        w[:,j+1] = c(w[:,j+1],t[j+1],p)  # condicion sobre w

    return t,w

## **Ejercicio 2)** Modelo de Hodgkin y Huxley

## TeorÃ­a

### Las neuronas son capacitores

SegÃºn el modelo de Hodgkin y Huxley, a primera vista las neuronas funcionan como capacitores evidenciando una diferencia de potencial elÃ©ctrico entre el exterior y el interior debido a las diferentes concentraciones de cargas iÃ³nicas que pueden existir entre estas dos regiones.
En el esquema, se utilizan agujas muy finas para inyectar una corriente iÃ³nica $i$ al interior de una neurona sumergida en una soluciÃ³n salina conductora.

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/esquema1.png?raw=true:, width=100" alt="My Image" width=300>

Recordemos que un capacitor sujeto a una diferencia de potencial $v:=v_A-v_B$ adquiere una carga $q=cv$.
La carga crece segÃºn $\dot{q}=i$, donde $i$ denota la corriente de carga elÃ©ctrica que va desde el exterior hacia el interior.

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/circuito1.png?raw=true:, width=100" alt="My Image" width=500>

### Las neuronas son capacitores con fugas

Si miramos un poco mÃ¡s en detalle, veremos que la membrana de las neuronas tienen canales por donde pueden circular iones.
Podemos pensar a los canales como agujeritos en la membrana por donde la carga se fuga, que podemos modelar como a una resistencia conectada en paralelo al capacitor.

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/circuito2.png?raw=true:, width=100" alt="My Image" width=500>

La corriente $i_g=g(v_A-v_B)=gv$, donde $i_g$ denota una corriente que va desde el interior hacia el exterior.

### Las neuronas tienen generadores de tensiÃ³n

Los canales pueden ser selectivos.
Por ejemplo, algunos canales sÃ³lo dejan pasar los iones $\mathrm{K}^+$.
Por ende, si inyectamos al interior de la neurona una mezcla de iones $\mathrm{K}^+$ y $\mathrm{Cl}^-$, de manera que la carga total se encuentre equilibrada, los iones $\mathrm{K}^+$ comenzarÃ¡n a difundir desde el interior hacia el exterior a travÃ©z de los canales selectivos, mientras que los de $\mathrm{Cl}^-$ se quedarÃ¡n en el interior, provocando un excedente de cargas negativas en el interior que crecen en el tiempo hasta que la diferencia de potencial emergente frena la corriente de iÃ³nes $\mathrm{K}^+$.

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/baterias.png?raw=true:, width=100" alt="My Image" width=500>

A este fenÃ³meno se lo denomina potencial de Nernst o de reversiÃ³n y se lo incorpora al circuito que modela la neurona como una baterÃ­a conectada en serie a la resistencia asociada a cada canal

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/circuito3.png?raw=true:, width=100" alt="My Image" width=500>

Ahora, la corriente es $i_g=g(v_A-v_C)=g(v_A-(v_B+e_g))=g(v_A-v_B-e_g)=g(v-e_g)=:g(v-v_g)$, donde en la Ãºltima iguadad reconocemos la definiciÃ³n del potencial de Nernst $v_g$.

Si la baterÃ­a se encuentra en sentido inverso

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/circuito4.png?raw=true:, width=100" alt="My Image" width=500>

entonces $i_g=g(v_A-v_C)=g(v_A-(v_B-e_g))=g(v_A-v_B+e_g)=g(v+e_g)=:g(v-v_g)$, por lo que el potencial de Nernst resulta negativo, $v_g=-e_g$.

### Algunos canales forman reÃ³statos (resistencias de conductancia variable)

Algunos canales tienen varias compuertas que se abren o cierran, permitiendo o impidiendo el paso de iones. Un mismo canal puede tener varias compuertas, y para que el canal sea conductor deben encontrarse abiertas las compuertas correspondientes.

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/compuertas.png?raw=true:, width=100" alt="My Image" width=500>

La conductividad asociada a estos canales, depende de la fracciÃ³n de compuertas abiertas en cada tipo de canal.
SegÃºn el modelo, la conductancia total asociada a los canales de $\mathrm{Na}$ se aproxima por

$$g_{\mathrm{Na}} = \bar{g}_{\mathrm{Na}}p_{\mathrm{Na}}$$

donde $\bar{g}_{\mathrm{Na}}$ es la conductancia de $\mathrm{Na}$ mÃ¡xima posible y

$$p_{\mathrm{Na}} = m^3h$$

es la fracciÃ³n de canales de $\mathrm{Na}$ que se encuentran en estado conductor.
AquÃ­ $m$ y $h$ son las fracciones de compuertas de activaciÃ³n e inactivaciÃ³n, respectivamente, que se encuentran abiertas.
Los canales de $\mathrm{Na}$ poseen 3 compuertas de activaciÃ³n y 1 de inactivaciÃ³n, por lo que, asumiendo que se abren de manera independiente, $m$ tiene un exponente 3 y $h$ un exponente 1.

De manera similar, la conductancia asociada a los canales de $\mathrm{K}$ se aproxima por

$$g_{\mathrm{K}} = \bar{g}_{\mathrm{K}}p_{\mathrm{K}}$$

donde $\bar{g}_{\mathrm{K}}$ es la conductancia de $\mathrm{K}$ mÃ¡xima posible,

$$p_{\mathrm{K}} = n^4$$

es la fracciÃ³n de canales de $\mathrm{K}$ abiertos y $n$ es la fracciÃ³n de compuertas abiertas en este tipo de canales.
Cada canal de $\mathrm{K}$ posee 4 compuertas de tipo $n$, de ahÃ­ el exponente 4.

Estas aproximaciones asumen que las compuertas abren o cierran de manera independiente, dependiendo de la diferencia de potencial de membrana existente $v$.
MÃ¡s precisamente, las fracciones de compuertas abiertas de cada tipo satisfacen

\begin{eqnarray}
\dot{n}(t)&=&\alpha_n(v(t))(1-n(t))-\beta_n(v(t)) n(t)\\
\dot{m}(t)&=&\alpha_m(v(t))(1-m(t))-\beta_m(v(t)) m(t)\\
\dot{h}(t)&=&\alpha_h(v(t))(1-h(t))-\beta_h(v(t)) h(t)
\end{eqnarray}

donde

\begin{eqnarray}
\alpha_h(v)&=&\frac{0.07}{ms}\,\exp\bigg(-\frac{v}{20\,mV}\bigg)\\
\alpha_m(v)&=&\frac{0.1}{ms\,mV}\frac{25\,mV-v}{\exp\big(\frac{25\,mV-v}{10\,mV}\big)-1}\\
\alpha_n(v)&=&\frac{0.01}{ms\,mV}\frac{10\,mV-v}{\exp\big(\frac{10\,mV-v}{10\,mV}\big)-1}\\
\end{eqnarray}

representan las tasas a las cuales las compuertas cerradas de cada tipo abren y

\begin{eqnarray}
\beta_h(v)&=&\frac{1}{ms}\frac{1}{\exp\big(\frac{30\,mV-v}{10\,mV}\big)+1}\\
\beta_m(v)&=&\frac{4}{ms}\exp\bigg(-\frac{v}{18\,mV}\bigg)\\
\beta_n(v)&=&\frac{0.125}{ms}\exp\bigg(-\frac{v}{80\,mV}\bigg)
\end{eqnarray}

representan las tasas a las cuales las compuertas abiertas de cada tipo cierran, respectivamente.

En resumen, al modelo de neurona de Hodgkin y Huxley se lo puede representar por el circuito

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/circuito5.png?raw=true:, width=100" alt="My Image" width=500>

en donde reconocemos la existencia de 3 canales iÃ³nicos: uno de pÃ©rdida (leak) de conductancia constante $g_l$, potencial de Nernst $v_l=e_l$ y por el que fluye una corriente $i_l$, uno de $\mathrm{Na}$ de conductancia variable $g_{\mathrm{Na}}$, potencial de Nernst $v_{\mathrm{Na}}=e_{\mathrm{Na}}$ y por el que fluye una corriente $i_{\mathrm{Na}}$, y uno de $\mathrm{K}$ de conductancia $g_{\mathrm{K}}$, potencial de Nernst $v_{\mathrm{K}}=-e_{\mathrm{K}}$ y por el que fluye una corriente $i_{\mathrm{K}}$.

Aplicando la ley de Kirchoff para corrientes, vemos que

$$i=i_{\mathrm{Na}}+i_{\mathrm{K}}+i_l+i_c$$

Es decir, la corriente $i$ inyectada al interior de la neurona se divide en dos partes.
Por un lado, en la corriente $i_m=i_{\mathrm{Na}}+i_{\mathrm{K}}+i_l$ que fluye a travÃ©z de los canales en la membrana,
y por otro lado, en la corriente $i_c$ que carga el capacitor.

Remplazando, obtenemos una ODE para el potencial de membrana

\begin{eqnarray}
i_c&=&i-i_{\mathrm{Na}}+i_{\mathrm{K}}+i_l\\
c\dot{v}&=&i-g_{\mathrm{Na}}(v-v_{\mathrm{Na}})-g_{\mathrm{K}}(v-v_{\mathrm{K}})-g_{l}(v-v_{l})\\
\dot{v}&=&c^{-1}(i-\bar{g}_{\mathrm{Na}}m^3h(v-v_{\mathrm{Na}})-\bar{g}_{\mathrm{K}}n^4(v-v_{\mathrm{K}})-g_{l}(v-v_{l}))
\end{eqnarray}

La Ãºltima lÃ­nea de la anterior ecuaciÃ³n, junto con las ODEs anteriormente mencionadas para $n$, $m$ y $h$, determinan el sistema de ODEs del modelo de Hodgkin y Huxley.

### Comportamiento cualitativo

Durante el proceso de disparo, diferentes compuertas se abren y cierran a diferentes tiempos.
El siguiente esquema resume el proceso en 4 etapas

<img src="https://github.com/jipphysics/redes-neuronales-2022/blob/master/practicos/assets/channels.png?raw=true:, width=100" alt="My Image" width=300>

Estimaciones de los tiempos carÃ¡cterÃ­sticos (de apertura/cierre) y los valores de equilibrio de las fracciones de compuertas abiertas de cada tipo que observan en las diferentes etapas, se pueden obtener de reescribir las ODEs para $n$, $m$ y $h$ de la forma

\begin{eqnarray}
\dot{n}(t)&=&\tau_n^{-1}(v(t))(n_{\infty}(v(t))-n(t))\\
\dot{m}(t)&=&\tau_m^{-1}(v(t))(m_{\infty}(v(t))-m(t))\\
\dot{h}(t)&=&\tau_h^{-1}(v(t))(h_{\infty}(v(t))-h(t))
\end{eqnarray}

donde

\begin{eqnarray}
\tau_n(v)&=&(\alpha_n(v)+\beta_n(v))^{-1}\\
\tau_m(v)&=&(\alpha_m(v)+\beta_m(v))^{-1}\\
\tau_h(v)&=&(\alpha_h(v)+\beta_h(v))^{-1}
\end{eqnarray}

y

\begin{eqnarray}
n_{\infty}(v)&=&\tau_n(v)\alpha_n(v)\\
m_{\infty}(v)&=&\tau_m(v)\alpha_m(v)\\
h_{\infty}(v)&=&\tau_h(v)\alpha_h(v)
\end{eqnarray}

Finalmente, listamos los valores de los parÃ¡metros del modelo (ver libro de Izhikevich)

* $c=1\,\mu F/cm^2$ : capacitancia de membrana
* $\bar{g}_{\mathrm{Na}}=120\,mS/cm^2$ : conductancia mÃ¡xima de $\mathrm{Na}$
* $\bar{g}_{\mathrm{K}}=36\,mS/cm^2$ : conductancia mÃ¡xima de $\mathrm{K}$
* $g_l=0.3\,mS/cm^2$ : conductancia mÃ¡xima de perdida
* $v_{\mathrm{Na}}=120\,mV$ : potencial de reversiÃ³n de la corriente de $\mathrm{Na}$
* $v_{\mathrm{K}}=-12\,mV$ : potencial de reversiÃ³n de la corriente de $\mathrm{K}$
* $v_l=10.6\,mV$ : potencial de reversiÃ³n de la corriente de perdida
* $i(t)\sim 10\,\mu A/cm^2$ : corriente de entrada al tiempo $t$
* $t \sim 5\,ms$ : tiempo

AclaraciÃ³n, los valores de los potenciales de Nernst se toman en referencia al potencial de equilibrio definido en $v\approx 0\, mV$.

## Ejercicio 2) implementaciÃ³n

**1)** Usando la secciÃ³n de teorÃ­a y las referencias, implemente el modelo de Hodgkin y Huxley el integrador numÃ©rico de ODEs tipo Runge-Kutta de orden 4 del prÃ¡ctico 4.

**2)** Grafique los valores de equilibrio de las distintas fracciones $n_{\infty}$, $m_{\infty}$ y $h_{\infty}$ de compuertas abiertas, en funciÃ³n de la diferencia de potencial de membrana $v$.

**3)** Grafique los tiempos caracterÃ­sticos $\tau_m$, $\tau_n$ y $\tau_h$ de apertura y cierre, asociados a los distintos tipos de compuertas, en funciÃ³n de la diferencia de potencial de membrana $v$.

In [ ]:
# 2.1, 2.2 y 2.3) Implementacion de Hodgkin & Huxley y analisis de compuertas
 `n # Parametros estandar del modelo HH (Izhikevich / Hodgkin & Huxley 1952)
 C_m = 1.0       # uF/cm^2
 g_Na_bar = 120.0 # mS/cm^2
 g_K_bar = 36.0   # mS/cm^2
 g_l = 0.3        # mS/cm^2
 v_Na = 120.0     # mV
 v_K = -12.0      # mV
 v_l = 10.6       # mV
 `n # Tasas de transicion (alpha y beta) con proteccion contra division por cero
 def alpha_h(v):
     return 0.07 * np.exp(-v / 20.0)
 `n def beta_h(v):
     return 1.0 / (np.exp((30.0 - v) / 10.0) + 1.0)
 `n def alpha_m(v):
     diff = 25.0 - v
     # Evitar indeterminacion 0/0 mediante aproximacion de Taylor si |diff| < 1e-7
     if np.isscalar(diff):
         if np.abs(diff) < 1e-7: return 1.0
         return 0.1 * diff / (np.exp(diff / 10.0) - 1.0)
     else:
         res = np.empty_like(diff, dtype=float)
         mask = np.abs(diff) < 1e-7
         res[mask] = 1.0
         res[~mask] = 0.1 * diff[~mask] / (np.exp(diff[~mask] / 10.0) - 1.0)
         return res
 `n def beta_m(v):
     return 4.0 * np.exp(-v / 18.0)
 `n def alpha_n(v):
     diff = 10.0 - v
     if np.isscalar(diff):
         if np.abs(diff) < 1e-7: return 0.1
         return 0.01 * diff / (np.exp(diff / 10.0) - 1.0)
     else:
         res = np.empty_like(diff, dtype=float)
         mask = np.abs(diff) < 1e-7
         res[mask] = 0.1
         res[~mask] = 0.01 * diff[~mask] / (np.exp(diff[~mask] / 10.0) - 1.0)
         return res
 `n def beta_n(v):
     return 0.125 * np.exp(-v / 80.0)
 `n # Valores estacionarios y constantes de tiempo
 def n_inf(v): return alpha_n(v) / (alpha_n(v) + beta_n(v))
 def m_inf(v): return alpha_m(v) / (alpha_m(v) + beta_m(v))
 def h_inf(v): return alpha_h(v) / (alpha_h(v) + beta_h(v))
 `n def tau_n(v): return 1.0 / (alpha_n(v) + beta_n(v))
 def tau_m(v): return 1.0 / (alpha_m(v) + beta_m(v))
 def tau_h(v): return 1.0 / (alpha_h(v) + beta_h(v))
 `n # Sistema de 4 ODEs: x = [v, n, m, h]
 def f_hh(x, t, p):
     v, n, m, h = x[0], x[1], x[2], x[3]
     i_inj = p["i_func"](t)
     i_Na = g_Na_bar * (m**3) * h * (v - v_Na)
     i_K = g_K_bar * (n**4) * (v - v_K)
     i_l = g_l * (v - v_l)
     
     dv_dt = (i_inj - i_Na - i_K - i_l) / C_m
     dn_dt = alpha_n(v) * (1.0 - n) - beta_n(v) * n
     dm_dt = alpha_m(v) * (1.0 - m) - beta_m(v) * m
     dh_dt = alpha_h(v) * (1.0 - h) - beta_h(v) * h
     return np.array([dv_dt, dn_dt, dm_dt, dh_dt])
 `n # 2.2 y 2.3) Graficar n_inf, m_inf, h_inf y tau vs potencial v
 v_grid = np.linspace(-40, 100, 500)
 fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 `n axes[0].plot(v_grid, n_inf(v_grid), label=r"$n_\infty(v)$ (activacion K)", color="blue", lw=2)
 axes[0].plot(v_grid, m_inf(v_grid), label=r"$m_\infty(v)$ (activacion Na)", color="red", lw=2)
 axes[0].plot(v_grid, h_inf(v_grid), label=r"$h_\infty(v)$ (inactivacion Na)", color="green", lw=2)
 axes[0].set_title("Ejercicio 2.2: Fracciones estacionarias de compuertas abiertas")
 axes[0].set_xlabel("Potencial $v$ [mV]")
 axes[0].set_ylabel("Fraccion abierta")
 axes[0].legend()
 axes[0].grid(True, alpha=0.3)
 `n axes[1].plot(v_grid, tau_n(v_grid), label=r"$\tau_n(v)$", color="blue", lw=2)
 axes[1].plot(v_grid, tau_m(v_grid), label=r"$\tau_m(v)$ (rapida)", color="red", lw=2)
 axes[1].plot(v_grid, tau_h(v_grid), label=r"$\tau_h(v)$", color="green", lw=2)
 axes[1].set_title("Ejercicio 2.3: Tiempos caracteristicos $\tau(v)$ [ms]")
 axes[1].set_xlabel("Potencial $v$ [mV]")
 axes[1].set_ylabel("Tiempo [ms]")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()


## Ejercicio 3) valores de equilibrio

**1)** Integre el sistema de ODEs del modelo de Hodgkin y Huxley sujeto a una corriente total $i(t)=0$ para todo $t$. Use la condiciÃ³n inicial $v_0=0mV$, $n_0=m_0=h_0=0$ y $t_0=0ms$. Integre hasta el tiempo final $t_f=500ms$ usando un paso temporal $dt=0.01ms$.

**2)** Grafique el potencial de membrana en funciÃ³n del tiempo, i.e. grafique $v(t)$ vs $t$ en el rango calculado.

**3)** Grafique las diferentes corrientes de iones cruzando la membrana en funciÃ³n del tiempo, i.e. grafique  la corriente de iones de sodio $i_{Na}(t)$, la corriente de iones de potasio $i_K(t)$ y la corriente de pÃ©rdida $i_l(t)$ vs $t$ en el rango calculado.

**4)** Grafique las fracciÃ³n de compuertas abiertas de cada tipo en funciÃ³n del tiempo, i.e. grafique $n(t)$, $m(t)$ y $h(t)$ vs $t$ en el rango calculado.

**5)** Almacene los valores de equilibrio a corriente nula de las variables dinÃ¡micas usando los valores de las mismas obtenidos a tiempos largos. Es decir, almacene los valores $v^* \approx v(t_f)$, $n^* \approx n(t_f)$, $m^* \approx m(t_f)$ y $h^* \approx h(t_f)$, para ser utilizado como condiciones iniciales en futuras integraciones de las ODEs del modelo de Hodgkin y Huxley.

In [ ]:
# 3.1 a 3.5) Integracion a corriente nula y extraccion de valores de equilibrio
 p_eq = {"i_func": lambda t: 0.0}
 xa_0 = np.array([0.0, 0.0, 0.0, 0.0]) # v=0, n=0, m=0, h=0
 dt = 0.01
 tf = 500.0
 k_steps = int(tf / dt)
 `n t_eq, w_eq = integrador_ode(rk4, f_hh, xa_0, 0.0, tf, k_steps, p_eq)
 `n # 3.5) Guardar valores de equilibrio a tiempos largos
 v_star = w_eq[0, -1]
 n_star = w_eq[1, -1]
 m_star = w_eq[2, -1]
 h_star = w_eq[3, -1]
 CI_eq = np.array([v_star, n_star, m_star, h_star])
 print(f"Valores de equilibrio: v* = {v_star:.3f} mV, n* = {n_star:.4f}, m* = {m_star:.4f}, h* = {h_star:.4f}")
 `n # Corrientes ionicas
 i_Na_eq = g_Na_bar * (w_eq[2, :]**3) * w_eq[3, :] * (w_eq[0, :] - v_Na)
 i_K_eq = g_K_bar * (w_eq[1, :]**4) * (w_eq[0, :] - v_K)
 i_l_eq = g_l * (w_eq[0, :] - v_l)
 `n fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
 axes[0].plot(t_eq, w_eq[0, :], color="black", lw=1.5)
 axes[0].axhline(v_star, color="red", linestyle="--", label=f"$v^* = {v_star:.2f}$ mV")
 axes[0].set_ylabel("Potencial $v$ [mV]")
 axes[0].set_title("Ejercicio 3: Relajacion al equilibrio a corriente nula ($i(t)=0$)")
 axes[0].legend()
 axes[0].grid(True, alpha=0.3)
 `n axes[1].plot(t_eq, i_Na_eq, label=r"$i_{Na}$", color="red", lw=1.2)
 axes[1].plot(t_eq, i_K_eq, label=r"$i_K$", color="blue", lw=1.2)
 axes[1].plot(t_eq, i_l_eq, label=r"$i_l$", color="green", lw=1.2)
 axes[1].set_ylabel("Corrientes [uA/cm^2]")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 `n axes[2].plot(t_eq, w_eq[1, :], label=r"$n(t)$", color="blue", lw=1.2)
 axes[2].plot(t_eq, w_eq[2, :], label=r"$m(t)$", color="red", lw=1.2)
 axes[2].plot(t_eq, w_eq[3, :], label=r"$h(t)$", color="green", lw=1.2)
 axes[2].set_xlabel("Tiempo $t$ [ms]")
 axes[2].set_ylabel("Fraccion abierta")
 axes[2].legend()
 axes[2].grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()


## Ejercicio 4) estÃ­mulo dÃ©bil y estÃ­mulo fuerte

**1)** Implemente una funciÃ³n de corriente de entrada o membrana dada por:

$$
i(t) = \left\{
\begin{array}{ll}
10 \mu A/cm^2, & t\in [2ms,2.5ms] \\
30 \mu A/cm^2, & t\in [10ms,10.5ms] \\
0 \mu A/cm^2, & c.c. \\
\end{array}
\right.
$$

AquÃ­, $c.c.$ indica *en caso contrario*. Esta corriente representa un estÃ­mulo dÃ©bil seguido de uno fuerte. GrafÃ­que la corriente $i(t)$ vs $t$ en el rango $t\in[0ms,20ms]$.

**2)** Integre nuevamente el sistema de ODEs del modelo de Hodgkin y Huxley sujeto a la corriente del inciso anterior. Use como condiciÃ³n inicial a tiempo $t=0$ los valores de equilibrio estimados en el ejercicio anterior. Integre hasta el tiempo final $t_f=500ms$ usando un paso temporal $dt=0.01ms$.

**3)** Grafique el potencial de membrana en funciÃ³n del tiempo, i.e. $v(t)$ vs $t$ en el rango calculado.

**4)** Grafique la evoluciÃ³n de las fraciones de compuertas abiertas, $n(t)$, $m(t)$ y $h(t)$ vs $t$.

**5)** Discuta como responde la neurona en el primer impulso a $t=2ms$. Luego, como responde al segundo impulso a $t=10ms$. Existe una diferencia? Explique.


In [ ]:
# 4.1 a 4.5) Estimulo debil (10 uA a los 2 ms) y fuerte (30 uA a los 10 ms)
 def i_pulsos(t):
     if 2.0 <= t <= 2.5:
         return 10.0
     elif 10.0 <= t <= 10.5:
         return 30.0
     else:
         return 0.0
 `n p_pulsos = {"i_func": i_pulsos}
 tf_p = 50.0 # visualizacion clara de los 2 estimulos
 kp = int(tf_p / dt)
 t_p, w_p = integrador_ode(rk4, f_hh, CI_eq, 0.0, tf_p, kp, p_pulsos)
 `n fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
 # Corriente inyectada
 i_vals_p = [i_pulsos(ti) for ti in t_p]
 axes[0].plot(t_p, i_vals_p, color="crimson", lw=2)
 axes[0].set_ylabel("Corriente $i(t)$ [uA/cm^2]")
 axes[0].set_title("Ejercicio 4: Estimulo subumbral ($t=2$ ms, 10 uA) vs supraumbral ($t=10$ ms, 30 uA)")
 axes[0].grid(True, alpha=0.3)
 `n # Potencial de membrana
 axes[1].plot(t_p, w_p[0, :], color="black", lw=1.8)
 axes[1].axhline(v_star, color="gray", linestyle=":", label="Reposo")
 axes[1].set_ylabel("Potencial $v(t)$ [mV]")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 `n # Compuertas ionicas
 axes[2].plot(t_p, w_p[1, :], label="n (activacion K)", color="blue", lw=1.5)
 axes[2].plot(t_p, w_p[2, :], label="m (activacion Na)", color="red", lw=1.5)
 axes[2].plot(t_p, w_p[3, :], label="h (inactivacion Na)", color="green", lw=1.5)
 axes[2].set_xlabel("Tiempo $t$ [ms]")
 axes[2].set_ylabel("Compuertas")
 axes[2].legend()
 axes[2].grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()
 `n print("""Discusion del Ejercicio 4.5:`n 1. En t=2 ms (estimulo debil de 10 uA): Se produce una despolarizacion pasiva subumbral. La compuerta m se abre ligeramente, pero la corriente de K+ y fuga superan a la corriente entrante de Na+, por lo que el potencial regresa suavemente al reposo sin disparar un potencial de accion.`n 2. En t=10 ms (estimulo fuerte de 30 uA): La corriente inyectada es suficiente para superar el umbral critico. Se desencadena el ciclo de retroalimentacion positiva de Hodgkin: m se abre rapidamente (~0.2 ms), permitiendo una entrada masiva de iones Na+ que despolariza la membrana hasta ~100 mV (potencial de accion). Luego, la compuerta h inactiva el Na+ y la compuerta lenta n abre los canales de K+, repolarizando e hiperpolarizando la celula.""")


## Ejercicio 5) rÃ¡faga

**1)** Implemente la corriente de membrana

$$
i(t) = \left\{
\begin{array}{ll}
10 \mu A/cm^2, & t\in [5ms,\infty ms) \\
0 \mu A/cm^2, & c.c. \\
\end{array}
\right.
$$

Esta corriente representa un estÃ­mulo constante.

**2)** Integre nuevamente las ODEs para $t\in[0ms,100ms]$, usando como condiciÃ³n incial los valores de equilibrio derivados en el inciso **6)** y un paso de integraciÃ³n $dt=0.01$.

**3)** Grafique nuevamente el potencial de membrana en el rango de tiempos calculado.

**4)** Grafique nuevamente fracciones de compuertas abiertas de cada tipo vs el tiempo.

**5)** Discuta lo que observa y explique.

In [ ]:
# 5.1 a 5.5) Estimulo constante sostenido (rafaga de potenciales de accion)
 def i_const_step(t):
     return 10.0 if t >= 5.0 else 0.0
 `n p_rafaga = {"i_func": i_const_step}
 tf_rafaga = 100.0
 k_rafaga = int(tf_rafaga / dt)
 t_raf, w_raf = integrador_ode(rk4, f_hh, CI_eq, 0.0, tf_rafaga, k_rafaga, p_rafaga)
 `n fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
 axes[0].plot(t_raf, w_raf[0, :], color="black", lw=1.5)
 axes[0].set_ylabel("Potencial $v(t)$ [mV]")
 axes[0].set_title("Ejercicio 5: Disparo ritmico sostenido (Tren de espigas) para $i(t) = 10$ uA/cm^2")
 axes[0].grid(True, alpha=0.3)
 `n axes[1].plot(t_raf, w_raf[1, :], label="n (K+)", color="blue", lw=1.5)
 axes[1].plot(t_raf, w_raf[2, :], label="m (Na+ act)", color="red", lw=1.5)
 axes[1].plot(t_raf, w_raf[3, :], label="h (Na+ inact)", color="green", lw=1.5)
 axes[1].set_xlabel("Tiempo $t$ [ms]")
 axes[1].set_ylabel("Compuertas")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()
 `n print("""Discusion del Ejercicio 5.5:`n Al inyectar una corriente continua sostenida i = 10 uA/cm^2 por encima del umbral, el sistema de Hodgkin-Huxley sufre una bifurcacion de Hopf supercritica. El punto fijo de reposo pierde estabilidad y nace un ciclo limite estable en el espacio de fases de 4 dimensiones, generando un tren periodico regular de potenciales de accion (frecuencia tipica ~60-70 Hz).""")


## Ejercicio 6) perÃ­odo refractario

**1)** Implemente la corriente de membrana

$$
i(t) = \left\{
\begin{array}{ll}
10 \mu A/cm^2, & t\in [10ms\, k,10 ms\, k + 2ms], k \in \{1,2,3,4,5,...\}\\
0 \mu A/cm^2, & c.c. \\
\end{array}
\right.
$$

**2)** Integre nuevamente las ODEs para $t\in[0ms,100ms]$, usando la corriente del inciso **12)**, la condiciÃ³n incial los valores de equilibrio derivados en el inciso **6)** y un paso de integraciÃ³n $dt=0.01$.

**3)** Grafique nuevamente el potencial de membrana en el rango de tiempos calculado.

**4)** Grafique nuevamente fracciones de canales activos e inactivos vs el tiempo.

**5)** Discuta lo que observa y explique.

In [ ]:
# 6.1 a 6.5) Periodo refractario con pulsos periodicos de 2 ms cada 10 ms
 def i_refractario(t):
     # Pulsos en [10k, 10k+2] para k=1, 2, 3...
     modulo = t % 10.0
     if t >= 10.0 and modulo <= 2.0:
         return 10.0
     return 0.0
 `n p_refr = {"i_func": i_refractario}
 tf_refr = 100.0
 k_refr = int(tf_refr / dt)
 t_refr, w_refr = integrador_ode(rk4, f_hh, CI_eq, 0.0, tf_refr, k_refr, p_refr)
 `n fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
 axes[0].plot(t_refr, w_refr[0, :], color="black", lw=1.5)
 axes[0].set_ylabel("Potencial $v(t)$ [mV]")
 axes[0].set_title("Ejercicio 6: Respuesta a pulsos periodicos repetidos (Analisis de refractariedad)")
 axes[0].grid(True, alpha=0.3)
 `n axes[1].plot(t_refr, w_refr[1, :], label="n (K+)", color="blue", lw=1.5)
 axes[1].plot(t_refr, w_refr[2, :], label="m (Na+ act)", color="red", lw=1.5)
 axes[1].plot(t_refr, w_refr[3, :], label="h (Na+ inact)", color="green", lw=1.5)
 axes[1].set_xlabel("Tiempo $t$ [ms]")
 axes[1].set_ylabel("Compuertas")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()
 `n print("""Discusion del Ejercicio 6.5:`n El periodo refractario absoluto (~1-2 ms) corresponde al intervalo donde la compuerta h esta completamente inactivada (h cercano a 0) y n muy alta; en este estado es imposible disparar un nuevo spike sin importar la intensidad del estimulo.`n El periodo refractario relativo (~5-15 ms) corresponde a la recuperacion gradual de h hacia su valor de reposo mientras n se desactiva lentamente; en esta fase se requiere una corriente significativamente mayor para alcanzar el umbral de disparo.""")


## Ejercicio 7) exitaciones espontÃ¡neas en respuesta al ruido

**1)** Implemente una corriente estocÃ¡stica que retorne un valor $i(t)\sim i_0 N(0,1)$ (i.e. $i_0$ por un valor aleatorio obtenido de una distribuciÃ³n normal de media 0 y varianza 1) para cada valor de $t$ en el que sea evaluada.

**2)** Integre nuevamente las ODEs para $t\in[0ms,500ms]$, usando la corriente del inciso **22)** para $i_0=50\mu A$, la condiciÃ³n incial los valores de equilibrio derivados en el inciso **6)** y un paso de integraciÃ³n $dt=0.01$.
**IMPORTANTE**: para integrar, utilice en este caso el mÃ©todo de Euler, ya que estarÃ­amos integrando un proceso estocÃ¡stico.

**3)** Grafique nuevamente el potencial de membrana en el rango de tiempos calculado.

**4)** Grafique nuevamente fracciones de compuertas abiertas de cada tipo vs el tiempo.

**5)** Observa picos de activaciÃ³n cada tanto? Aparecen con regularidad? Estime con que frecuencia observa los picos.

In [ ]:
# 7.1 a 7.5) Excitaciones espontaneas en respuesta a corriente estocastica (Ruido Gaussiano)
 np.random.seed(42)
 i0_ruido = 50.0  # uA/cm^2
 tf_ruido = 500.0
 k_ruido = int(tf_ruido / dt)
 `n # Pre-generar realizacion del proceso de ruido Gaussiano blanco N(0, 1)
 ruido_array = np.random.normal(0.0, 1.0, k_ruido + 1)
 `n def f_hh_stoch(x, t, p):
     step_idx = int(round(t / dt))
     step_idx = min(step_idx, k_ruido)
     i_t = p["i0"] * ruido_array[step_idx]
     
     v, n, m, h = x[0], x[1], x[2], x[3]
     i_Na = g_Na_bar * (m**3) * h * (v - v_Na)
     i_K = g_K_bar * (n**4) * (v - v_K)
     i_l = g_l * (v - v_l)
     
     dv_dt = (i_t - i_Na - i_K - i_l) / C_m
     dn_dt = alpha_n(v) * (1.0 - n) - beta_n(v) * n
     dm_dt = alpha_m(v) * (1.0 - m) - beta_m(v) * m
     dh_dt = alpha_h(v) * (1.0 - h) - beta_h(v) * h
     return np.array([dv_dt, dn_dt, dm_dt, dh_dt])
 `n p_stoch = {"i0": i0_ruido}
 # Integracion con Euler para proceso estocastico (Euler-Maruyama)
 t_ruido, w_ruido = integrador_ode(euler, f_hh_stoch, CI_eq, 0.0, tf_ruido, k_ruido, p_stoch)
 `n # Deteccion de picos espontaneos (v > 50 mV)
 v_traza = w_ruido[0, :]
 spikes_idx = []
 for j in range(1, len(v_traza) - 1):
     if v_traza[j] > 50.0 and v_traza[j] > v_traza[j-1] and v_traza[j] > v_traza[j+1]:
         if len(spikes_idx) == 0 or (t_ruido[j] - t_ruido[spikes_idx[-1]]) > 5.0:
             spikes_idx.append(j)
 `n num_spikes = len(spikes_idx)
 freq_spont = num_spikes / (tf_ruido / 1000.0) # Hz
 print(f"Picos detectados: {num_spikes} en {tf_ruido} ms => Frecuencia media = {freq_spont:.1f} Hz")
 `n fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
 axes[0].plot(t_ruido, v_traza, color="black", lw=1.2)
 axes[0].scatter(t_ruido[spikes_idx], v_traza[spikes_idx], color="red", zorder=5, label=f"Disparos espontaneos ({num_spikes})")
 axes[0].set_ylabel("Potencial $v(t)$ [mV]")
 axes[0].set_title(f"Ejercicio 7: Disparos inducidos por ruido estocastico ($i_0={i0_ruido}$ uA/cm^2, Euler)")
 axes[0].legend()
 axes[0].grid(True, alpha=0.3)
 `n axes[1].plot(t_ruido, w_ruido[1, :], label="n (K+)", color="blue", lw=1.2)
 axes[1].plot(t_ruido, w_ruido[2, :], label="m (Na+ act)", color="red", lw=1.2)
 axes[1].plot(t_ruido, w_ruido[3, :], label="h (Na+ inact)", color="green", lw=1.2)
 axes[1].set_xlabel("Tiempo $t$ [ms]")
 axes[1].set_ylabel("Compuertas")
 axes[1].legend()
 axes[1].grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()
 `n print("""Discusion del Ejercicio 7.5:`n Se observan picos de activacion de forma intermitente (proceso de escape inducido por ruido tipo Kramers). Los picos NO aparecen con perfecta regularidad, sino siguiendo una distribucion estocastica de intervalos entre disparos (Poissoniana modulada por refractariedad). Las fluctuaciones Gaussianas suficientemente grandes cruzan transitoriamente la separatriz del sistema, disparando la trayectoria no lineal del potencial de accion.""")
